In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/aisehack-theme-2/test_in/psfc.npy
/kaggle/input/competitions/aisehack-theme-2/test_in/t2.npy
/kaggle/input/competitions/aisehack-theme-2/test_in/SO2.npy
/kaggle/input/competitions/aisehack-theme-2/test_in/NMVOC_finn.npy
/kaggle/input/competitions/aisehack-theme-2/test_in/bio.npy
/kaggle/input/competitions/aisehack-theme-2/test_in/rain.npy
/kaggle/input/competitions/aisehack-theme-2/test_in/u10.npy
/kaggle/input/competitions/aisehack-theme-2/test_in/swdown.npy
/kaggle/input/competitions/aisehack-theme-2/test_in/pblh.npy
/kaggle/input/competitions/aisehack-theme-2/test_in/v10.npy
/kaggle/input/competitions/aisehack-theme-2/test_in/cpm25.npy
/kaggle/input/competitions/aisehack-theme-2/test_in/NH3.npy
/kaggle/input/competitions/aisehack-theme-2/test_in/q2.npy
/kaggle/input/competitions/aisehack-theme-2/test_in/NMVOC_e.npy
/kaggle/input/competitions/aisehack-theme-2/test_in/NOx.npy
/kaggle/input/competitions/aisehack-theme-2/test_in/PM25.npy
/kaggle/input/competit

In [ ]:
# ─────────────────────────────────────────────
# CELL 1: Imports & Config
# ─────────────────────────────────────────────
import os, gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DATA = '/kaggle/input/competitions/aisehack-theme-2'
TEMP = '/kaggle/temp'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Features to use — start with this high-signal subset
MET = ['u10', 'v10', 'pblh', 'rain', 't2', 'q2']
EMIS = ['PM25', 'SO2', 'NOx']
FEATURES = ['cpm25'] + MET + EMIS   # 10 features total
N_FEATURES = len(FEATURES)

T_IN_CPM = 10     # cpm25 history available at test time
T_IN_MET = 26     # met/emis full window
T_OUT = 16        # forecast horizon
STRIDE = 2        # adjust based on compute budget
MONTHS = ['APRIL_16', 'JULY_16', 'OCT_16', 'DEC_16']
VAL_MONTH = 'OCT_16'   # hold out for validation


In [ ]:
# ─────────────────────────────────────────────
# CELL 2: Normalization Stats (Grid-wise)
# ─────────────────────────────────────────────
def compute_stats(months):
    """Compute per-gridpoint mean/std across training months."""
    stats = {}
    for feat in FEATURES:
        arrays = []
        for m in months:
            path = os.path.join(DATA, 'raw', m, f'{feat}.npy')
            arr = np.load(path).astype(np.float32)  # (T, 140, 124)
            arrays.append(arr)
        concat = np.concatenate(arrays, axis=0)  # (T_total, 140, 124)
        stats[feat] = {
            'mean': concat.mean(axis=0),   # (140, 124)
            'std':  concat.std(axis=0) + 1e-8
        }
        del concat; gc.collect()
    return stats

train_months = [m for m in MONTHS if m != VAL_MONTH]
stats = compute_stats(train_months)
np.save(os.path.join(TEMP, 'norm_stats.npy'), stats)
print("Stats computed for:", train_months)


In [ ]:
# ─────────────────────────────────────────────
# CELL 3: Sample Construction
# ─────────────────────────────────────────────
def build_samples(months, stride, stats):
    """
    Returns:
      X: (N, T_IN_MET, H, W, C) — all features, padded cpm25 for t>10
      y: (N, H, W, T_OUT)       — future cpm25 (denormalized ground truth)
    """
    X_list, y_list = [], []
    for m in months:
        raw = {}
        for feat in FEATURES:
            arr = np.load(os.path.join(DATA, 'raw', m, f'{feat}.npy')).astype(np.float32)
            raw[feat] = (arr - stats[feat]['mean']) / stats[feat]['std']
        
        T = raw['cpm25'].shape[0]
        window = T_IN_MET + T_OUT  # 26 + 16 = 42... but data is per-month
        # Use 26-step input, 16-step output: total window = 42
        # We take input t=0..25, output t=10..25 (future 16 steps after t=9)
        # Actually: input context = first 26h, output = hours 10..25 (16 steps)
        # This matches test_in structure: cpm25 has 10h, met has 26h
        
        for i in range(0, T - T_IN_MET - T_OUT + 1, stride):
            # Input: 26 hours of all features
            inp = np.stack([raw[feat][i:i+T_IN_MET] for feat in FEATURES], axis=-1)  # (26, 140, 124, C)
            
            # Mask cpm25 to only first 10h (simulate test conditions)
            inp[T_IN_CPM:, :, :, 0] = 0.0
            
            # Output: next 16 hours of cpm25 in physical units
            cpm_raw = np.load(os.path.join(DATA, 'raw', m, 'cpm25.npy')).astype(np.float32)
            out_raw = cpm_raw[i + T_IN_MET : i + T_IN_MET + T_OUT]  # (16, 140, 124)
            out = out_raw.transpose(1, 2, 0)  # (140, 124, 16)
            
            X_list.append(inp)
            y_list.append(out)
    
    return np.stack(X_list), np.stack(y_list)

X_train, y_train = build_samples(train_months, stride=STRIDE, stats=stats)
X_val, y_val = build_samples([VAL_MONTH], stride=STRIDE*2, stats=stats)
print(f"Train: {X_train.shape}, Val: {X_val.shape}")
# Expected: X ~ (N, 26, 140, 124, 10), y ~ (N, 140, 124, 16)


In [ ]:
# ─────────────────────────────────────────────
# CELL 4: Dataset & DataLoader
# ─────────────────────────────────────────────
class PM25Dataset(Dataset):
    def __init__(self, X, y):
        # Pre-convert to tensors on CPU to avoid repeated conversion
        self.X = torch.from_numpy(X)  # (N, T, H, W, C)
        self.y = torch.from_numpy(y)  # (N, H, W, T_OUT)
    
    def __len__(self): return len(self.X)
    
    def __getitem__(self, idx):
        x = self.X[idx].permute(3, 0, 1, 2)  # (C, T, H, W)
        return x, self.y[idx]

train_ds = PM25Dataset(X_train, y_train)
val_ds   = PM25Dataset(X_val,   y_val)

train_dl = DataLoader(train_ds, batch_size=4, shuffle=True,
                      num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=8, shuffle=False,
                      num_workers=2, pin_memory=True)


In [ ]:
# ─────────────────────────────────────────────
# CELL 5: TFNO2D Model
# ─────────────────────────────────────────────
class SpectralConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, modes1, modes2):
        super().__init__()
        self.modes1, self.modes2 = modes1, modes2
        scale = 1 / (in_ch * out_ch)
        self.weights = nn.Parameter(
            scale * torch.randn(in_ch, out_ch, modes1, modes2, 2)
        )
    
    def compl_mul2d(self, x, w):
        # x: (B, C_in, M1, M2, 2) as complex via view
        xr, xi = x[..., 0], x[..., 1]
        wr, wi = w[..., 0], w[..., 1]
        return torch.stack([xr*wr - xi*wi, xr*wi + xi*wr], dim=-1)
    
    def forward(self, x):
        B, C, H, W = x.shape
        x_ft = torch.fft.rfft2(x, norm='ortho')  # (B, C, H, W//2+1) complex
        # Truncate to modes
        out_ft = torch.zeros(B, self.weights.shape[1], H, W//2+1,
                             dtype=torch.cfloat, device=x.device)
        x_ft_real = torch.view_as_real(x_ft[:, :, :self.modes1, :self.modes2])
        out_real  = self.compl_mul2d(x_ft_real, self.weights)
        out_ft[:, :, :self.modes1, :self.modes2] = torch.view_as_complex(out_real)
        return torch.fft.irfft2(out_ft, s=(H, W), norm='ortho')

class FNOBlock(nn.Module):
    def __init__(self, width, modes1, modes2):
        super().__init__()
        self.spectral = SpectralConv2d(width, width, modes1, modes2)
        self.bypass   = nn.Conv2d(width, width, 1)
        self.norm     = nn.GroupNorm(8, width)
    
    def forward(self, x):
        return F.gelu(self.norm(self.spectral(x) + self.bypass(x)))

class TFNO2D(nn.Module):
    """
    Tucker-factorized FNO2D for PM2.5 forecasting.
    Input:  (B, C*T_in, H, W) — channels = features × time steps flattened
    Output: (B, T_out, H, W)
    """
    def __init__(self, in_channels, out_steps=16, width=64, modes=24, depth=6):
        super().__init__()
        self.lift = nn.Conv2d(in_channels, width, 1)
        self.blocks = nn.ModuleList([FNOBlock(width, modes, modes) for _ in range(depth)])
        self.proj = nn.Sequential(
            nn.Conv2d(width, width * 2, 1),
            nn.GELU(),
            nn.Conv2d(width * 2, out_steps, 1)
        )
    
    def forward(self, x):
        # x: (B, C, T, H, W) → flatten C,T → (B, C*T, H, W)
        B, C, T, H, W = x.shape
        x = x.reshape(B, C*T, H, W)
        x = self.lift(x)
        for block in self.blocks:
            x = block(x)
        x = self.proj(x)      # (B, T_out, H, W)
        return x.permute(0, 2, 3, 1)  # (B, H, W, T_out)

model = TFNO2D(
    in_channels=N_FEATURES * T_IN_MET,  # 10 * 26 = 260
    out_steps=T_OUT,
    width=64,
    modes=20,
    depth=4
).to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


In [ ]:
# ─────────────────────────────────────────────
# CELL 6: Metric-Aligned Loss + Training
# ─────────────────────────────────────────────
def metric_loss(pred, target):
    # pred, target: (B, H, W, T) — matches evaluation metric structure
    spatial_mse = torch.mean((pred - target) ** 2, dim=(1, 2))  # (B, T)
    return torch.mean(torch.sqrt(spatial_mse + 1e-8))

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=1e-3,
    steps_per_epoch=len(train_dl), epochs=30,
    pct_start=0.1  # warmup
)

best_val_loss = float('inf')
for epoch in range(30):
    model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        pred = model(xb)
        loss = metric_loss(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
    
    model.eval()
    val_losses = []
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred = model(xb)
            val_losses.append(metric_loss(pred, yb).item())
    
    val_loss = np.mean(val_losses)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), '/kaggle/temp/best_model.pt')
    
    if epoch % 5 == 0:
        print(f"Epoch {epoch:3d} | Val RMSE: {val_loss:.4f} | Best: {best_val_loss:.4f}")


In [ ]:
# ─────────────────────────────────────────────
# CELL 7: Inference + Save preds.npy
# ─────────────────────────────────────────────
model.load_state_dict(torch.load('/kaggle/temp/best_model.pt'))
model.eval()

# Load test inputs
test_inputs = []
for feat in FEATURES:
    path = os.path.join(DATA, 'test_in', f'{feat}.npy')
    arr = np.load(path).astype(np.float32)  # (996, T_feat, 140, 124)
    # Normalize with training stats
    mean = stats[feat]['mean'][None, None]  # (1, 1, 140, 124)
    std  = stats[feat]['std'][None, None]
    arr_norm = (arr - mean) / std
    # Pad cpm25 to T_IN_MET timesteps (zero-pad future)
    if feat == 'cpm25':
        pad = np.zeros((996, T_IN_MET - T_IN_CPM, 140, 124), dtype=np.float32)
        arr_norm = np.concatenate([arr_norm, pad], axis=1)  # (996, 26, 140, 124)
    test_inputs.append(arr_norm)

X_test = np.stack(test_inputs, axis=-1)  # (996, 26, 140, 124, 10)
print("Test input shape:", X_test.shape)

all_preds = []
test_ds = torch.from_numpy(X_test)
batch_size = 16
with torch.no_grad():
    for i in range(0, len(test_ds), batch_size):
        batch = test_ds[i:i+batch_size].permute(0, 4, 1, 2, 3).to(DEVICE)  # (B, C, T, H, W)
        pred = model(batch)  # (B, H, W, T_OUT) — normalized
        # Denormalize: grid-wise
        cpm_mean = torch.from_numpy(stats['cpm25']['mean']).to(DEVICE)  # (140, 124)
        cpm_std  = torch.from_numpy(stats['cpm25']['std']).to(DEVICE)
        pred = pred * cpm_std[None, :, :, None] + cpm_mean[None, :, :, None]
        pred = torch.clamp(pred, min=0.0)
        all_preds.append(pred.cpu().numpy())

preds = np.concatenate(all_preds, axis=0).astype(np.float32)
print("Output shape:", preds.shape)  # Must be (996, 140, 124, 16)
assert preds.shape == (996, 140, 124, 16)
assert np.isfinite(preds).all()

np.save('/kaggle/working/preds.npy', preds)
print("✅ preds.npy saved!")
